# Stage 9E — Compare two models using validation only

**Goal:** Compare a majority-rule baseline, logistic regression, and a tree model on the new daily-coverage dataset. Choose a model and a probability threshold using January–February 2026 validation.

**Why:** More data and a different model are separate questions. This experiment tests both on the same frozen split. We will emphasize recall with F2, while requiring at least 0.60 validation precision when possible. F2 favors recall more than precision.

**Important:** Do not load or score the March–April test file in this stage. The chosen design will be refit and tested once in Stage 9F. The 0.60 precision floor is a classroom policy for review and planning, not a city requirement.

In [ ]:
# Cell 2 — Why: verify and load only train and validation.
from google.colab import drive
from pathlib import Path
import hashlib, json, pandas as pd, numpy as np, time

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
split_id = "20260926T142339294958Z"
split_dir = root / "splits" / split_id
split = json.loads((split_dir / "manifest.json").read_text())
if split["split_id"] != split_id or not split["test_sealed_for_model_selection"]:
    raise ValueError("Wrong split or unsealed test")
frames = {}
for name in ["train", "validation"]:
    file = split_dir / f"{name}.jsonl"
    raw = file.read_bytes()
    if hashlib.sha256(raw).hexdigest() != split["files"][name]["sha256"]:
        raise ValueError(f"{name} file changed")
    frames[name] = pd.read_json(file, lines=True)
    if len(frames[name]) != split["files"][name]["rows"]:
        raise ValueError(f"{name} row count changed")
train, valid = frames["train"], frames["validation"]
features = split["features"]
required = set(features) | {"label", "unique_key", "created_date"}
for name, frame in frames.items():
    if set(frame.columns) != required or not frame["label"].isin([0, 1]).all():
        raise ValueError(f"{name} has invalid columns or labels")
print("Train:", len(train), "Validation:", len(valid))
print("Test remains unopened")

In [ ]:
# Cell 3 — Why: check winter coverage and a simple baseline.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score

def metrics(y, pred):
    return {"accuracy": round(accuracy_score(y, pred), 3),
            "precision": round(precision_score(y, pred, zero_division=0), 3),
            "recall": round(recall_score(y, pred, zero_division=0), 3),
            "f1": round(f1_score(y, pred, zero_division=0), 3),
            "f2": round(fbeta_score(y, pred, beta=2, zero_division=0), 3)}

common = int(train["label"].mode().iloc[0])
baseline = metrics(valid["label"], np.full(len(valid), common))
print("Train Snow or Ice rows:", int(train["complaint_type"].eq("Snow or Ice").sum()))
print("Validation Snow or Ice rows:", int(valid["complaint_type"].eq("Snow or Ice").sum()))
print("Previously unseen validation complaint types:",
      len(set(valid["complaint_type"]) - set(train["complaint_type"])))
print("Majority baseline:", common, baseline)

In [ ]:
# Cell 4 — Why: define two different model designs with fixed settings.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

text_steps = Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="MISSING")),
                       ("encode", OneHotEncoder(handle_unknown="ignore"))])
time_steps = Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                       ("encode", OneHotEncoder(handle_unknown="ignore"))])
one_hot = ColumnTransformer([("text", text_steps, ["complaint_type", "borough"]),
                             ("time", time_steps, ["month", "hour", "weekday"])])
models = {
    "logistic": Pipeline([("prepare", one_hot),
                           ("model", LogisticRegression(solver="liblinear", max_iter=1000,
                                                        random_state=42))]),
    "tree": Pipeline([("prepare", OrdinalEncoder(handle_unknown="use_encoded_value",
                                                  unknown_value=np.nan, max_categories=200)),
                      ("model", HistGradientBoostingClassifier(
                          categorical_features=[True] * 5, max_iter=100,
                          max_leaf_nodes=31, min_samples_leaf=50,
                          early_stopping=False, random_state=42))])
}
print("Candidates:", list(models))

In [ ]:
# Cell 5 — Why: fit each candidate on train only and score validation.
probabilities = {}
fit_seconds = {}
for name, model in models.items():
    print("Fitting:", name, flush=True)
    started = time.monotonic()
    model.fit(train[features], train["label"])
    fit_seconds[name] = round(time.monotonic() - started, 1)
    probabilities[name] = model.predict_proba(valid[features])[:, 1]
    print(name, "fit seconds:", fit_seconds[name],
          "validation at 0.50:", metrics(valid["label"], probabilities[name] >= 0.50),
          flush=True)

In [ ]:
# Cell 6 — Why: choose the threshold using validation, not test.
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
results = []
for name, probs in probabilities.items():
    for threshold in thresholds:
        score = metrics(valid["label"], probs >= threshold)
        results.append({"model": name, "threshold": threshold, **score})
eligible = [row for row in results if row["precision"] >= 0.60]
selection_pool = eligible if eligible else results
chosen = max(selection_pool, key=lambda row: (row["f2"], row["precision"]))
print("Validation choices (sorted by F2):")
for row in sorted(results, key=lambda row: row["f2"], reverse=True):
    print(row)
print("Chosen:", chosen)
print("Precision floor met:", bool(eligible))

In [ ]:
# Cell 7 — Why: inspect important groups before final testing.
chosen_pred = probabilities[chosen["model"]] >= chosen["threshold"]
checks = {"Snow or Ice": valid["complaint_type"].eq("Snow or Ice"),
          "HEAT/HOT WATER": valid["complaint_type"].eq("HEAT/HOT WATER"),
          "BROOKLYN": valid["borough"].eq("BROOKLYN")}
validation_groups = {}
for name, mask in checks.items():
    y = valid.loc[mask, "label"]
    pred = chosen_pred[mask.to_numpy()]
    late = int(y.sum())
    validation_groups[name] = {"rows": int(mask.sum()), "late_cases": late,
                               "recall": round(recall_score(y, pred), 3) if late else None,
                               "precision": round(precision_score(y, pred), 3)
                               if int(pred.sum()) else None}
    print(name, validation_groups[name])

In [ ]:
# Cell 8 — Why: save candidates and the decision before opening test.
from datetime import datetime, timezone
import joblib, platform, sklearn

experiment_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
experiment_dir = root / "experiments" / experiment_id
experiment_dir.mkdir(parents=True, exist_ok=False)
artifacts = {}
for name, model in models.items():
    file = experiment_dir / f"{name}.joblib"
    joblib.dump(model, file)
    artifacts[name] = {"file": file.name, "sha256": hashlib.sha256(file.read_bytes()).hexdigest()}
experiment = {"experiment_id": experiment_id, "split_id": split_id,
              "source_prepared_id": split["source_prepared_id"],
              "features": features, "baseline_validation": baseline,
              "selection_rule": "highest validation F2 with precision >= 0.60 when possible",
              "thresholds_tried": thresholds, "all_validation_results": results,
              "chosen": chosen, "precision_floor_met": bool(eligible),
              "chosen_validation_groups": validation_groups,
              "fit_seconds": fit_seconds, "artifacts": artifacts,
              "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                           "numpy": np.__version__, "scikit_learn": sklearn.__version__,
                           "joblib": joblib.__version__}, "lesson_version": "stage9e-v1"}
(experiment_dir / "experiment.json").write_text(json.dumps(experiment, indent=2) + "\n", encoding="utf-8")
print("Experiment ID:", experiment_id)
print("Chosen model and threshold:", chosen["model"], chosen["threshold"])
print("Test still sealed")

# Stage 9E takeaway and quiz

**What you learned:** The same wider data can be used to compare different model designs fairly. We trained on 2025 only, chose the model and threshold on January–February 2026, checked important validation groups, and saved the candidates and decision before test evaluation.

**Easy quiz:**

1. Why compare both models on the same validation period?
2. What does lowering the probability threshold usually do to recall and precision?
3. Why look at Snow or Ice and HEAT/HOT WATER separately?
4. Why save the chosen settings before opening the test file?

Send me **Train Snow or Ice rows**, **Majority baseline**, both **validation at 0.50** scores, **Chosen**, group results, and **Experiment ID**. Stage 9F will refit the chosen design and use the sealed test once.